# Fusión de etiquetas manuales

Lee:

- Todos los **CSV** en `data/etiquetas_manuales`, **excluyendo** `validacion_CONTR_edwin(2374).csv`
- Los Excel **`BACK_data.xlsx`** y **`METH_data.xlsx`** en la misma carpeta

Conserva solo las columnas que aparecen en **todos** esos archivos (por ejemplo, `article_content` solo está en los `.xlsx` y no se incluye en el unificado), concatena las filas y exporta el resultado a CSV y Excel.

In [1]:
from pathlib import Path

import pandas as pd

# Raíz del proyecto (directorio padre de notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "etiquetas_manuales"
EXCLUDE_CSV = "validacion_CONTR_edwin(2374).csv"
EXTRA_XLSX_FILENAMES = ("BACK_data.xlsx", "METH_data.xlsx")

OUTPUT_CSV = PROJECT_ROOT / "data" / "etiquetas_manuales" / "etiquetas_manuales_unificadas.csv"
OUTPUT_XLSX = PROJECT_ROOT / "data" / "etiquetas_manuales" / "etiquetas_manuales_unificadas.xlsx"

In [2]:
csv_paths = sorted(
    p for p in DATA_DIR.glob("*.csv")
    if p.name != EXCLUDE_CSV
)
xlsx_paths = [DATA_DIR / name for name in EXTRA_XLSX_FILENAMES]

if not csv_paths:
    raise FileNotFoundError(f"No hay CSV en {DATA_DIR} (tras excluir {EXCLUDE_CSV})")

missing_xlsx = [p for p in xlsx_paths if not p.exists()]
if missing_xlsx:
    raise FileNotFoundError(
        "Faltan Excel esperados en la carpeta: "
        + ", ".join(p.name for p in missing_xlsx)
    )

input_paths = sorted(
    [*csv_paths, *xlsx_paths],
    key=lambda p: (p.suffix.lower(), p.name.lower()),
)

print(f"Archivos a fusionar ({len(input_paths)}):")
for p in input_paths:
    print(f"  - {p.name}")

Archivos a fusionar (5):
  - validacion_CONC_edwin(80).csv
  - validacion_LIM_edwin(40).csv
  - validated_results_cristian.csv
  - validated_results_natalia.csv
  - validated_results_pablo.csv


In [3]:
frames = []
column_sets = []

for path in input_paths:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, dtype=str)
    elif path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, dtype=str, engine="openpyxl")
    else:
        raise ValueError(f"Formato no soportado: {path}")

    df.columns = df.columns.str.strip()
    frames.append(df)
    column_sets.append(set(df.columns))

common_columns = sorted(set.intersection(*column_sets))

print("Columnas por archivo:")
for path, df in zip(input_paths, frames):
    extra = sorted(set(df.columns) - set(common_columns))
    missing = sorted(set(common_columns) - set(df.columns))
    print(f"  {path.name}: {len(df.columns)} cols", end="")
    if extra:
        print(f" | solo en este: {extra}", end="")
    if missing:
        print(f" | faltantes vs comunes: {missing}", end="")
    print()

print(f"\nColumnas comunes a todos ({len(common_columns)}): {common_columns}")

Columnas por archivo:
  validacion_CONC_edwin(80).csv: 7 cols
  validacion_LIM_edwin(40).csv: 7 cols
  validated_results_cristian.csv: 7 cols
  validated_results_natalia.csv: 7 cols
  validated_results_pablo.csv: 7 cols

Columnas comunes a todos (7): ['category', 'confidence', 'key_phrases', 'keywords', 'snippet', 'source_file', 'source_path']


In [5]:
partes = [df[common_columns].copy() for df in frames]
df_unificado = pd.concat(partes, ignore_index=True)

print(df_unificado.shape)
df_unificado.head()

(1034, 7)


,category,confidence,key_phrases,keywords,snippet,source_file,source_path
0,CONC,high,una transformación cultural; productos softwar...,"devops, entrega, software, práctica, prácticas...",dicho cambio implica una transformación cultur...,103759022.txt,D:\santi\Documents\MAIA\Bimestre 2026-11\Proye...
1,CONC,high,tratamiento quirúrgico; turf toe; distintas af...,"tratamiento, turf, tener, patrón, forma, trata...",Había bastante controversia a la hora de estab...,107292875.txt,D:\santi\Documents\MAIA\Bimestre 2026-11\Proye...
2,CONC,high,discapacidad intelectual; hospitales psiquiátr...,"discapacidad, persona, personas, cosa, además,...",Además de que me encantaría poder demostrarle ...,107294272.txt,D:\santi\Documents\MAIA\Bimestre 2026-11\Proye...
3,CONC,high,la figura delde; la pianista acompañante; artí...,"investigación, españa, carencia, estudios, dis...","En España, además, los estudios de investigaci...",107420336.txt,D:\santi\Documents\MAIA\Bimestre 2026-11\Proye...
4,CONC,high,metal porcelana; prótesis parcial; base metáli...,"tratamiento, plan, sesión, prótesis, pronóstic...",F. PRONÓSTICO El pronóstico se valora mediante...,108649676.txt,D:\santi\Documents\MAIA\Bimestre 2026-11\Proye...


In [ ]:
df_unificado.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
df_unificado.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")

print(f"Guardado CSV:  {OUTPUT_CSV}")
print(f"Guardado XLSX: {OUTPUT_XLSX}")